In [1]:
import os

import numpy as np
import pandas as pd

import torch
from torch import nn

In [2]:
import sys, os
sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import parse_network, io
from scLEMBAS.model.scl import SignalingModel
from scLEMBAS.model.tfa import TFA
from scLEMBAS.model.train import TrainSimple, TrainCat

/nobackup/users/hmbaghda/Software/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
n_cores = 12
os.environ["OMP_NUM_THREADS"] = str(n_cores)
os.environ["MKL_NUM_THREADS"] = str(n_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cores)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(n_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(n_cores)

seed = 888

device = "cuda" if torch.cuda.is_available() else "cpu"

data_path = '/nobackup/users/hmbaghda/scLEMBAS/analysis'
models_path = os.path.join(data_path, 'processed', 'models')
if not os.path.isdir(models_path):
    os.mkdir(models_path)

In [4]:
tf_adata = io.read_tfad(file_name = os.path.join(data_path, 'processed', 'ID_tf_activity.h5ad'))

/nobackup/users/hmbaghda/Software/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.4.0 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [5]:
source_label = 'source_genesymbol'
target_label = 'target_genesymbol'
weight_label = 'mode_of_action'
stimulation_label = 'consensus_stimulation'
inhibition_label = 'consensus_inhibition'

In [6]:
from scLEMBAS.io import write_pickled_object, read_pickled_object
# sn_ppis = parse_network.load_network('omnipath', organism = 'mouse', static = True)
# sn_ppis = parse_network.extract_network(sn_ppis, curation_effort_thresh = 5, n_references_thresh = 3,
#                                         resources = ['HuRI','IntAct','KEGG-MEDICUS','NetPath','Reactome_SignaLink3','SPIKE','SignaLink3','SIGNOR', 
#                                                 'Baccin2019', 'Ramilowski2015', 'Reactome_LRdb', 'UniProt_LRdb', 'CellChatDB', 'CellPhoneDB', 'connectomeDB2020', 'scConnect'], 
#                                         drop_self = True, verbose = True)

# tf_labels = tf_adata.var.index.unique().tolist()

# ligand_labels = tf_adata.obs['sample'].unique().tolist()
# ligand_labels = [(l[0] + l[1:].lower()).replace('-', '') for l in ligand_labels] # mouse naming convention

# # filter for paths b/w ligand and tf
# fn_1 = parse_network.fully_connected_network(sn_ppis, ligand_labels, tf_labels, source_label = source_label, target_label = target_label, 
#                        path_finder = 'shortest')
# fn_2 = parse_network.fully_connected_network(sn_ppis, ligand_labels, tf_labels, source_label = source_label, target_label = target_label, 
#                        path_finder = 'connected')
# # of the methods to identify paths, retain the one that has the most interactions
# if fn_1.shape[0] > fn_2.shape[0]:
#     sn_ppis = fn_1
# else:
#     sn_ppis = fn_2

# del fn_1, fn_2

# sn_ppis.loc[sn_ppis[(sn_ppis[stimulation_label] == 1) & (sn_ppis[inhibition_label] == 1)].index, 
#     [stimulation_label, inhibition_label]] = [False, False]
# sn_ppis = parse_network.format_network(sn_ppis, weight_label, stimulation_label, inhibition_label)

# all_nodes = sn_ppis[source_label].tolist() + sn_ppis[target_label].tolist()
# input_ligands_available = sorted(set(ligand_labels).intersection(all_nodes))

# subset_tf = tf_adata[tf_adata.obs.TF_clusters.isin(['9', '15'])]
# sample_size = subset_tf.obs.TF_clusters.value_counts().min()
# large_cluster = subset_tf.obs.TF_clusters.value_counts().idxmax()
# small_cluster = subset_tf.obs.TF_clusters.value_counts().idxmin()
# large_cluster_barcodes = subset_tf.obs[subset_tf.obs.TF_clusters == large_cluster].index
# small_cluster_barcodes = subset_tf.obs[subset_tf.obs.TF_clusters == small_cluster].index.tolist()
# np.random.seed(seed)
# lcb_sub = list(np.random.choice(large_cluster_barcodes, sample_size, replace = False))
# subset_tf = subset_tf[lcb_sub + small_cluster_barcodes, :]
# subset_tf.obs.TF_clusters.value_counts()

# np.random.seed(seed)
# selected_ligand = np.random.choice(input_ligands_available, 1)[0]

# ligand_input = pd.DataFrame(subset_tf.obs.TF_clusters.map({'9': 0, '15': 1}))
# ligand_input.columns = [selected_ligand]
# tf_output = pd.DataFrame(subset_tf.X, index = subset_tf.obs.index, columns = subset_tf.var.index)

# mod_in = [sn_ppis, ligand_input, tf_output, subset_tf]
# write_pickled_object(mod_in, 'trash.pickle')
mod_in = read_pickled_object('trash.pickle')
sn_ppis, ligand_input, tf_output, subset_tf = mod_in

In [7]:
# linear scaling of inputs/outputs
projection_amplitude_in = 3
projection_amplitude_out = 1.2
# other parameters
bionet_params = {'target_steps': 100, 
                 'max_steps': 120, 
                 'exp_factor':50, 
                 'tolerance': 1e-5, 
                 'leak':1e-2, 
                'cat_max_norm': 1} # fed directly to model

# training parameters
lr_params = {'max_epochs': 100, 
             'learning_rate': 2e-3, 
             'reset_optimizer_epoch': 200}

other_params = {'batch_size': 256, 
                'network_noise_scale': 10, 
                'gradient_noise_scale': 1e-9}

regularization_params = {'param_lambda_L2': 1e-6, 
                         'discriminator_lambda_L2': 1e-5,
                         'moa_lambda_L1': 0.1, #'ligand_lambda_L2': 1e-5, 
                         'uniform_lambda_L2': 1e-4,  
                         'uniform_max': (1/1.2), 
                         'spectral_loss_factor': 1e-5}

spectral_radius_params = {'n_probes_spectral': 5, 
                          'power_steps_spectral': 50, 
                          'subset_n_spectral': 10}
training_params = {**lr_params, **other_params, **regularization_params, **spectral_radius_params}
target_spectral_radius = 0.8

# parameters for constructing the discriminator
discriminator_params = {'batch_momentum': 0.01,
                        'layer_norm': False,
                        'dropout_rate': 0.1,
                        'activation_fn': torch.nn.modules.activation.ReLU,
                        'n_hidden_nodes': [16, 16, 16]}

# # TFA
# encoder_dist = None
# decoder_dist = 'gauss'

# e_hp = {'batch_momentum': 0.01, 'layer_norm': False, 'dropout_rate': 0.1, 
#          'activation_fn': torch.nn.ReLU, 'n_hidden_nodes': [64], 'linear_output': True}
# d_hp = {'batch_momentum': 0.01, 'layer_norm': False, 'dropout_rate': 0.1, 
#          'activation_fn': torch.nn.ReLU, 'n_hidden_nodes': [64], 'linear_output': True}
# ve_hp = {'batch_momentum': 0.01, 'layer_norm': False, 'dropout_rate': 0.1, 
#          'activation_fn': nn.ReLU, 'n_hidden_nodes': [64], 'var_min': 1e-4}
# vd_hp = {'batch_momentum': 0.01, 'layer_norm': False, 'dropout_rate': 0.1, 
#          'activation_fn': nn.ReLU, 'n_hidden_nodes': [64], 'var_min': 1e-4}

# Hyper_params = {
#     'tfa': {'n_latent': 32, 'cat_max_norm': 1, 'encoder_dist': encoder_dist, 'decoder_dist': decoder_dist}, 
#     'encoder': ve_hp if encoder_dist == 'guass' else e_hp,
#     'decoder': vd_hp if decoder_dist == 'guass' else d_hp,
#     'discriminator': {'batch_momentum': 0.01, 'layer_norm': False, 'dropout_rate': 0.1, 'activation_fn': nn.ReLU, 'n_hidden_nodes': [16, 16, 16], 'return_logits': True}

# }

In [8]:
cat_keys = ['TF_clusters', 'celltype']
mod = SignalingModel(net = sn_ppis,
                     X_in = ligand_input,
                     y_out = tf_output, 
                     activation_function = 'MML', 
#                      covariates = subset_tf.obs,
#                      categorical_covariate_keys = cat_keys,
                     projection_amplitude_in = projection_amplitude_in, projection_amplitude_out = projection_amplitude_out,
                     weight_label = weight_label, source_label = source_label, target_label = target_label,
                     bionet_params = bionet_params,
                     dtype = torch.float32, device = device, seed = seed)

In [13]:
self = mod.signaling_network
X_in = mod.df_to_tensor(mod.X_in)
X_full = mod.input_layer(X_in) # transform to full network with ligand input concentrations


In [15]:
self.bias_basal.shape

torch.Size([332, 1])

In [18]:
X_full.T.shape

torch.Size([332, 2336])

In [19]:
X_full

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], grad_fn=<CopySlices>)

In [32]:
self.bias_basal[12:14]

tensor([[0.0010],
        [1.0000]], grad_fn=<SliceBackward0>)

In [31]:
(X_full.T + self.bias_basal)[12:14, :2]

tensor([[0.0010, 0.0010],
        [1.0000, 1.0000]], grad_fn=<SliceBackward0>)

In [36]:
bias_tot = self.bias_basal
X_bias = X_full.T + bias_tot # this is the bias with the projection_amplitude included
X_new = torch.zeros_like(X_bias) #initialize hidden state values at 0

In [ ]:
X_old = X_new

In [37]:
X_new = torch.mm(self.weights, X_new) # scale matrix by edge weights

In [40]:
self.weights.shape

torch.Size([332, 332])

In [ ]:
bias_tot = self.bias_basal
X_bias = X_full.T + bias_tot # this is the bias with the projection_amplitude included
X_new = torch.zeros_like(X_bias) #initialize hidden state values at 0

for t in range(self.bionet_params['max_steps']): # like an RNN, updating from previous time step
    X_old = X_new
    X_new = torch.mm(self.weights, X_new) # scale matrix by edge weights
    X_new = X_new + X_bias  # add original values and bias       
    X_new = self.activation(X_new, self.bionet_params['leak'])

    if (t % 10 == 0) and (t > 20):
        diff = torch.max(torch.abs(X_new - X_old))    
        if diff.lt(self.bionet_params['tolerance']):
            break

Y_full = X_new.T
return Y_full

In [11]:

Y_full = mod.signaling_network(X_full, covariates_idx) # train signaling network weights

(2336, 1)

# Start dev

In [9]:
# # define inputs
# X_in = mod.df_to_tensor(mod.X_in)
# covariates_idx = mod.signaling_network.covariates_to_tensor(sample_ids = mod.X_in.index)

# X_full = mod.input_layer(X_in) # transform to full network with ligand input concentrations
# Y_full = mod.signaling_network(X_full, covariates_idx) # train signaling network weights
# Y_hat = mod.output_layer(Y_full)

In [10]:
trainer = TrainCat(mod = mod,
                   prediction_optimizer = torch.optim.Adam,
                   prediction_loss_fn = torch.nn.MSELoss(reduction='mean'),
                   hyper_params = training_params,
                   train_split_frac = {'train': 0.8, 'test': 0.2, 'validation': None}, 
                   train_seed = seed)
self = trainer

/home/hmbaghda/Projects/scLEMBAS/scLEMBAS/model/train.py:115: UserWarning: Recommended to run `self.mod.signaling_network.prescale_weights()` prior to training
  warnings.warn('Recommended to run `self.mod.signaling_network.prescale_weights()` prior to training')


In [11]:
res = trainer.train_model()

  1%|▍                                          | 1/100 [00:00<01:07,  1.46it/s]

i=0, l=1.19894, s=0.285, r=0.00020, v=0


  2%|▊                                          | 2/100 [00:01<00:58,  1.67it/s]

i=1, l=1.19598, s=0.274, r=0.00020, v=0


  3%|█▎                                         | 3/100 [00:01<00:55,  1.74it/s]

...

 55%|███████████████████████                   | 55/100 [00:29<00:24,  1.85it/s]

i=54, l=0.94816, s=0.438, r=0.00021, v=2


 56%|███████████████████████▌                  | 56/100 [00:30<00:23,  1.84it/s]

i=55, l=0.94668, s=0.449, r=0.00021, v=2


KeyboardInterrupt: 

In [ ]:
# trainer = TrainSimple(mod = mod,
#                       prediction_optimizer = torch.optim.Adam,
#                       prediction_loss_fn = torch.nn.MSELoss(reduction='mean'),
#                       hyper_params = training_params,
#                       train_split_frac = {'train': 0.8, 'test': 0.2, 'validation': None}, 
#                       train_seed = seed)
# res = trainer.train_model()

# End dev

In [ ]:
# mod.input_layer.weights.requires_grad = False # don't learn scaling factors for the ligand input concentrations
# mod.signaling_network.prescale_weights(target_radius = target_spectral_radius) # spectral radius

# # trainer = TrainSimple(mod = mod,
# #                       prediction_optimizer = torch.optim.Adam,
# #                       prediction_loss_fn = torch.nn.MSELoss(reduction='mean'),
# #                       hyper_params = training_params,
# #                       train_split_frac = {'train': 0.8, 'test': 0.2, 'validation': None}, 
# #                       train_seed = seed)


# trainer = TrainCat(mod = mod,
#                    prediction_optimizer = torch.optim.Adam,
#                    prediction_loss_fn = torch.nn.MSELoss(reduction='mean'),
#                    hyper_params = training_params,
#                    train_split_frac = {'train': 0.8, 'test': 0.2, 'validation': None}, 
#                    train_seed = seed)


# trainer = TrainSC(mod = mod,
#                    prediction_optimizer = torch.optim.Adam,
#                    prediction_loss_fn = torch.nn.MSELoss(reduction='mean'),
#                    discriminator_optimizer = torch.optim.Adam,
#                    hyper_params = training_params,
#                    discriminator_params = discriminator_params,
#                    train_split_frac = {'train': 0.8, 'test': 0.2, 'validation': None}, 
#                    train_seed = seed)

# # res = trainer.train_model()